# 🏦 Loan Approval Prediction — EDA + ML + Model Export
> **Dataset**: 4269 loan applications | **Task**: Binary Classification (Approved / Rejected)

---
### Table of Contents
1. Setup & Data Loading
2. Exploratory Data Analysis (EDA)
3. Feature Engineering & Preprocessing
4. Model Training & Comparison
5. Best Model — Tuning & Evaluation
6. Export Model for Streamlit

## 1. Setup & Data Loading

In [ ]:
# Install required libraries
!pip install -q xgboost lightgbm shap

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, roc_auc_score, roc_curve,
                             ConfusionMatrixDisplay)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import shap
import joblib

# Plot style
plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')
PALETTE = ['#2ecc71', '#e74c3c']

print('Libraries loaded ✅')

In [ ]:
# ── Upload your CSV file ──
from google.colab import files
uploaded = files.upload()   # select  loan_approval_dataset.csv

import io
filename = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[filename]))

# Clean column names
df.columns = df.columns.str.strip()

# Strip whitespace from string columns
for col in df.select_dtypes('object').columns:
    df[col] = df[col].str.strip()

print(f'Shape: {df.shape}')
df.head()

## 2. Exploratory Data Analysis (EDA)

### 2.1 Basic Info

In [ ]:
print('=== Data Types & Non-Null Counts ===')
df.info()

In [ ]:
print('=== Descriptive Statistics ===')
df.describe(include='all').T

In [ ]:
print('=== Missing Values ===')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.any() else 'No missing values ✅')

In [ ]:
print('=== Duplicate Rows ===')
print(f'Duplicates: {df.duplicated().sum()}')

### 2.2 Target Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df['loan_status'].value_counts()
axes[0].bar(counts.index, counts.values, color=PALETTE, edgecolor='white', width=0.5)
axes[0].set_title('Loan Status — Count', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 20, str(v), ha='center', fontweight='bold')

axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=PALETTE, startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Loan Status — Share', fontsize=14, fontweight='bold')

plt.suptitle('Target Variable Distribution', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(counts.to_string())
print(f'\nClass balance ratio: {counts.min()/counts.max():.2f}')

### 2.3 Categorical Features Analysis

In [ ]:
cat_cols = ['education', 'self_employed']
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col in zip(axes, cat_cols):
    ct = pd.crosstab(df[col], df['loan_status'], normalize='index') * 100
    ct.plot(kind='bar', ax=ax, color=PALETTE, edgecolor='white', rot=0)
    ax.set_title(f'{col.replace("_"," ").title()} vs Loan Status', fontsize=13, fontweight='bold')
    ax.set_ylabel('Percentage (%)')
    ax.set_xlabel('')
    ax.legend(title='Status')
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))

plt.suptitle('Categorical Features vs Loan Status', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

### 2.4 Numerical Features Distribution

In [ ]:
num_cols = ['no_of_dependents', 'income_annum', 'loan_amount', 'loan_term',
            'cibil_score', 'residential_assets_value', 'commercial_assets_value',
            'luxury_assets_value', 'bank_asset_value']

fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    for status, color in zip(['Approved', 'Rejected'], PALETTE):
        subset = df[df['loan_status'] == status][col]
        axes[i].hist(subset, bins=30, alpha=0.6, color=color, label=status, edgecolor='none')
    axes[i].set_title(col.replace('_', ' ').title(), fontsize=11, fontweight='bold')
    axes[i].set_xlabel('')
    axes[i].legend(fontsize=8)

plt.suptitle('Numerical Feature Distributions by Loan Status', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

### 2.5 CIBIL Score — Deep Dive (Most Important Feature)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# KDE
for status, color in zip(['Approved', 'Rejected'], PALETTE):
    subset = df[df['loan_status'] == status]['cibil_score']
    axes[0].hist(subset, bins=40, alpha=0.55, color=color, label=status, density=True)
axes[0].axvline(x=600, color='navy', linestyle='--', linewidth=1.5, label='Score=600')
axes[0].set_title('CIBIL Score Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('CIBIL Score')
axes[0].legend()

# Boxplot
approved = df[df['loan_status'] == 'Approved']['cibil_score']
rejected = df[df['loan_status'] == 'Rejected']['cibil_score']
bp = axes[1].boxplot([approved, rejected], labels=['Approved', 'Rejected'],
                      patch_artist=True, notch=True)
for patch, color in zip(bp['boxes'], PALETTE):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[1].set_title('CIBIL Score Boxplot', fontsize=13, fontweight='bold')
axes[1].set_ylabel('CIBIL Score')

plt.tight_layout()
plt.show()

print('CIBIL Score Statistics by Status:')
print(df.groupby('loan_status')['cibil_score'].describe().round(1))

### 2.6 Correlation Heatmap

In [ ]:
df_corr = df.copy()
df_corr['loan_status_enc'] = (df_corr['loan_status'] == 'Approved').astype(int)

corr_cols = num_cols + ['loan_status_enc']
corr_matrix = df_corr[corr_cols].corr()

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
fig, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, linewidths=0.5, ax=ax,
            cbar_kws={'shrink': 0.8})
ax.set_title('Correlation Heatmap (Numerical Features)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 2.7 Dependents & Loan Term Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# No. of dependents
ct1 = pd.crosstab(df['no_of_dependents'], df['loan_status'])
ct1.plot(kind='bar', ax=axes[0], color=PALETTE, edgecolor='white', rot=0)
axes[0].set_title('Dependents vs Loan Status', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Number of Dependents')
axes[0].set_ylabel('Count')
axes[0].legend(title='Status')

# Loan term
ct2 = pd.crosstab(df['loan_term'], df['loan_status'])
ct2.plot(kind='bar', ax=axes[1], color=PALETTE, edgecolor='white', rot=0)
axes[1].set_title('Loan Term (Years) vs Status', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Loan Term (Years)')
axes[1].set_ylabel('Count')
axes[1].legend(title='Status')

plt.tight_layout()
plt.show()

### 2.8 Pairplot (Key Features)

In [ ]:
key_features = ['cibil_score', 'income_annum', 'loan_amount', 'loan_status']
pairplot_df = df[key_features].copy()
g = sns.pairplot(pairplot_df, hue='loan_status', palette={'Approved': '#2ecc71', 'Rejected': '#e74c3c'},
                 plot_kws={'alpha': 0.4, 's': 15}, diag_kind='kde')
g.fig.suptitle('Pairplot of Key Features', y=1.02, fontsize=14, fontweight='bold')
plt.show()

## 3. Feature Engineering & Preprocessing

In [ ]:
df_ml = df.copy()

# Drop loan_id (not a feature)
df_ml.drop(columns=['loan_id'], inplace=True)

# Encode target
df_ml['loan_status'] = (df_ml['loan_status'] == 'Approved').astype(int)

# Encode categorical features
df_ml['education'] = (df_ml['education'] == 'Graduate').astype(int)
df_ml['self_employed'] = (df_ml['self_employed'] == 'Yes').astype(int)

# Feature Engineering
df_ml['total_assets'] = (df_ml['residential_assets_value'] +
                         df_ml['commercial_assets_value'] +
                         df_ml['luxury_assets_value'] +
                         df_ml['bank_asset_value'])

df_ml['loan_to_income_ratio'] = df_ml['loan_amount'] / (df_ml['income_annum'] + 1)
df_ml['asset_to_loan_ratio']  = df_ml['total_assets'] / (df_ml['loan_amount'] + 1)
df_ml['income_per_dependent'] = df_ml['income_annum'] / (df_ml['no_of_dependents'] + 1)

print('Features after engineering:')
print(df_ml.columns.tolist())
print(f'\nShape: {df_ml.shape}')

In [ ]:
X = df_ml.drop(columns=['loan_status'])
y = df_ml['loan_status']

FEATURE_NAMES = X.columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
print(f'Train class balance: {y_train.value_counts(normalize=True).round(3).to_dict()}')

## 4. Model Training & Comparison

In [ ]:
models = {
    'Logistic Regression':    LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree':          DecisionTreeClassifier(random_state=42),
    'Random Forest':          RandomForestClassifier(n_estimators=200, random_state=42),
    'Gradient Boosting':      GradientBoostingClassifier(n_estimators=200, random_state=42),
    'XGBoost':                XGBClassifier(n_estimators=200, use_label_encoder=False,
                                             eval_metric='logloss', random_state=42),
    'LightGBM':               LGBMClassifier(n_estimators=200, random_state=42, verbose=-1),
    'SVM':                    SVC(probability=True, random_state=42)
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []

for name, model in models.items():
    X_tr = X_train_sc if name in ['Logistic Regression', 'SVM'] else X_train
    X_te = X_test_sc  if name in ['Logistic Regression', 'SVM'] else X_test

    cv_scores = cross_val_score(model, X_tr, y_train, cv=cv, scoring='accuracy')
    model.fit(X_tr, y_train)
    test_acc  = accuracy_score(y_test, model.predict(X_te))
    test_auc  = roc_auc_score(y_test, model.predict_proba(X_te)[:, 1])

    results.append({
        'Model': name,
        'CV Accuracy (mean)': round(cv_scores.mean(), 4),
        'CV Std': round(cv_scores.std(), 4),
        'Test Accuracy': round(test_acc, 4),
        'Test AUC-ROC': round(test_auc, 4)
    })
    print(f'{name:<25} | CV: {cv_scores.mean():.4f} ± {cv_scores.std():.4f} | Test Acc: {test_acc:.4f} | AUC: {test_auc:.4f}')

results_df = pd.DataFrame(results).sort_values('Test Accuracy', ascending=False)
results_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Test Accuracy
order = results_df['Model'].tolist()
colors = ['#2ecc71' if i == 0 else '#3498db' for i in range(len(order))]
bars = axes[0].barh(order, results_df['Test Accuracy'], color=colors, edgecolor='white')
axes[0].set_xlim(0.8, 1.01)
axes[0].set_title('Test Accuracy by Model', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Accuracy')
for bar, val in zip(bars, results_df['Test Accuracy']):
    axes[0].text(val + 0.001, bar.get_y() + bar.get_height()/2,
                  f'{val:.4f}', va='center', fontsize=9)

# AUC-ROC
bars2 = axes[1].barh(order, results_df['Test AUC-ROC'], color=colors, edgecolor='white')
axes[1].set_xlim(0.8, 1.01)
axes[1].set_title('AUC-ROC by Model', fontsize=13, fontweight='bold')
axes[1].set_xlabel('AUC-ROC')
for bar, val in zip(bars2, results_df['Test AUC-ROC']):
    axes[1].text(val + 0.001, bar.get_y() + bar.get_height()/2,
                  f'{val:.4f}', va='center', fontsize=9)

plt.suptitle('Model Comparison', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Best Model — Tuning & Evaluation

In [ ]:
# XGBoost / LightGBM are typically best — hyperparameter tuning
param_grid = {
    'n_estimators':   [200, 400],
    'max_depth':      [4, 6, 8],
    'learning_rate':  [0.05, 0.1],
    'subsample':      [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

xgb_base = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
grid_search = GridSearchCV(xgb_base, param_grid, cv=5, scoring='accuracy',
                            n_jobs=-1, verbose=1)
grid_search.fit(X_train, y_train)

print(f'\nBest params: {grid_search.best_params_}')
print(f'Best CV score: {grid_search.best_score_:.4f}')

In [ ]:
best_model = grid_search.best_estimator_
y_pred     = best_model.predict(X_test)
y_proba    = best_model.predict_proba(X_test)[:, 1]

print('=== BEST MODEL EVALUATION ===')
print(f'Test Accuracy : {accuracy_score(y_test, y_pred):.4f}')
print(f'AUC-ROC       : {roc_auc_score(y_test, y_proba):.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['Rejected', 'Approved']))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['Rejected', 'Approved'])
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix', fontsize=13, fontweight='bold')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_proba)
auc = roc_auc_score(y_test, y_proba)
axes[1].plot(fpr, tpr, color='#3498db', linewidth=2, label=f'AUC = {auc:.4f}')
axes[1].plot([0, 1], [0, 1], color='gray', linestyle='--', linewidth=1)
axes[1].set_title('ROC Curve', fontsize=13, fontweight='bold')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend()

# Feature Importance
fi = pd.Series(best_model.feature_importances_, index=FEATURE_NAMES).sort_values(ascending=True).tail(15)
fi.plot(kind='barh', ax=axes[2], color='#3498db', edgecolor='white')
axes[2].set_title('Top Feature Importances', fontsize=13, fontweight='bold')
axes[2].set_xlabel('Importance Score')

plt.tight_layout()
plt.show()

In [ ]:
# SHAP Values
explainer   = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_test)

plt.figure(figsize=(10, 7))
shap.summary_plot(shap_values, X_test, feature_names=FEATURE_NAMES, show=False)
plt.title('SHAP Feature Importance Summary', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Export Model for Streamlit Deployment

In [ ]:
import joblib

# Save model and feature names
joblib.dump(best_model,    'loan_model.pkl')
joblib.dump(scaler,        'scaler.pkl')
joblib.dump(FEATURE_NAMES, 'feature_names.pkl')

print('Saved: loan_model.pkl, scaler.pkl, feature_names.pkl ✅')

# Download all
from google.colab import files
files.download('loan_model.pkl')
files.download('scaler.pkl')
files.download('feature_names.pkl')

print('\n✅ Download complete! Add these 3 files to your GitHub repo.')

---
### ✅ Next Steps
1. Download `loan_model.pkl`, `scaler.pkl`, `feature_names.pkl`
2. Add them to your GitHub repository along with `app.py` and `requirements.txt`
3. Deploy on Streamlit Cloud at https://share.streamlit.io

See the `app.py` in the repo for the full Streamlit app!